# PDF Parsing hàng loạt: MarkItDown + Qwen2.5-VL-7B (GPU)

**Mục tiêu:** tự động quét **toàn bộ file PDF** trong một thư mục dataset trên Kaggle,
và với mỗi file:

1. Trích xuất nhanh bằng `markitdown` (baseline).
2. Phát hiện trang "khó" (ít text / scan / cần OCR lại).
3. Dùng Qwen2.5-VL-7B để OCR lại các trang khó đó (giữ bảng markdown, công thức LaTeX,
   giữ đúng dấu tiếng Việt).
4. Ghép kết quả theo đúng thứ tự trang, xuất ra 1 file `.md` riêng cho mỗi PDF.

Model chỉ được load **một lần duy nhất** và tái sử dụng cho tất cả các file, tránh
lãng phí thời gian/VRAM.

**Yêu cầu:** Bật GPU trong Kaggle: `Settings > Accelerator > GPU`.


## Bước 0 — Kiểm tra GPU

In [ ]:
!nvidia-smi

## Bước 1 — Cài đặt thư viện

**Lưu ý quan trọng:** Kaggle tự động import sẵn nhiều thư viện (PIL, torchvision...)
ngay khi khởi động notebook, trước cả khi bạn chạy ô cài đặt bên dưới. Nếu ô này nâng
cấp các thư viện đó, Python vẫn giữ bản cũ trong bộ nhớ cho đến khi kernel được khởi
động lại, gây ra các lỗi khó hiểu kiểu `ImportError: cannot import name '_Ink'`.

→ **Sau khi chạy xong ô cài đặt, PHẢI restart kernel** (ô "Bước 1.1" ngay bên dưới sẽ
tự làm việc này) trước khi chạy tiếp các bước sau.


In [ ]:
!pip install -q 'markitdown[pdf]' pymupdf tqdm
!pip install -q "transformers>=4.49.0,<4.52.0" accelerate qwen-vl-utils
!pip install -q -U bitsandbytes
print("Cài đặt xong. Hãy chạy ô Bước 1.1 bên dưới để restart kernel.")


### Bước 1.1 — Restart kernel để áp dụng thay đổi

Chạy ô bên dưới, kernel sẽ tự tắt và Kaggle tự khởi động lại (mất khoảng 5-10 giây,
có thể thấy vòng xoay loading). **Sau khi kernel khởi động lại xong, chạy tiếp từ
Bước 2** — không cần chạy lại Bước 1 hay ô này lần nữa.


In [ ]:
import os
os.kill(os.getpid(), 9)

## Bước 2 — Quét toàn bộ file PDF trong thư mục dataset

> **Lưu ý quan trọng về lỗi tên file tiếng Việt trên Kaggle:**
> Khi bạn upload trực tiếp từng file PDF lên Kaggle Dataset qua trình duyệt, Kaggle sẽ **tự động lọc bỏ các nguyên âm có dấu tiếng Việt** trong tên file (ví dụ: `Mô hình chú ý ngữ cảnh đa tầm nhìn...` bị biến thành `M hnh ch  ng cnh a tm nhn...`).
> 
> **2 giải pháp đã được tích hợp sẵn trong notebook:**
> 1. **(Khuyên dùng khi đưa dữ liệu lên Kaggle)**: Nén các file PDF thành 1 file `.zip` (hoặc `.tar.gz`) trên máy tính trước, rồi upload file zip đó lên Kaggle. Ô bên dưới **tự động phát hiện và giải nén file zip**, bảo toàn 100% tên file tiếng Việt có dấu!
> 2. **Nếu đã upload trực tiếp khiến tên file bị mất dấu:** Notebook tích hợp thuật toán **tự động đọc trang 1 của PDF** bằng PyMuPDF để trích xuất lại chính xác Tiêu đề bài báo tiếng Việt (dựa trên font chữ lớn nhất). Tên thư mục kết quả, file `source_meta.json`, và file Markdown đầu ra (đặt tên theo slug tiêu đề) sẽ tự động mang tên chuẩn theo tiêu đề tiếng Việt thực sự của tài liệu.


In [ ]:
import os
import zipfile
import tarfile
import fitz  # PyMuPDF

# Thư mục chứa dataset trên Kaggle
# Mặc định quét /kaggle/input (tự động nhận diện mọi dataset được gắn vào notebook)
DATASET_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working/all_result"
EXTRACTED_DIR = "/kaggle/working/extracted_pdfs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EXTRACTED_DIR, exist_ok=True)

# 1. Tự động kiểm tra và giải nén các file .zip/.tar nếu có trong dataset
# (Upload file .zip lên Kaggle là cách tốt nhất để bảo toàn 100% tên file tiếng Việt có dấu)
zip_extracted_count = 0
for root, dirs, files in os.walk(DATASET_DIR):
    for f in files:
        f_path = os.path.join(root, f)
        if f.lower().endswith(".zip"):
            try:
                with zipfile.ZipFile(f_path, 'r') as zf:
                    for member in zf.infolist():
                        try:
                            # Hỗ trợ giải nén file zip tiếng Việt từ Windows (CP437 -> UTF-8)
                            fname = member.filename.encode('cp437').decode('utf-8')
                        except Exception:
                            fname = member.filename
                        target_path = os.path.join(EXTRACTED_DIR, fname)
                        if member.is_dir():
                            os.makedirs(target_path, exist_ok=True)
                        else:
                            os.makedirs(os.path.dirname(target_path), exist_ok=True)
                            with zf.open(member) as sf, open(target_path, "wb") as df:
                                df.write(sf.read())
                    print(f"Đã giải nén zip: {f} -> {EXTRACTED_DIR}")
                    zip_extracted_count += 1
            except Exception as e:
                print(f"Lỗi khi giải nén {f}: {e}")
        elif f.lower().endswith((".tar.gz", ".tgz", ".tar")):
            try:
                with tarfile.open(f_path, 'r:*') as tf:
                    tf.extractall(EXTRACTED_DIR)
                    print(f"Đã giải nén tar: {f} -> {EXTRACTED_DIR}")
                    zip_extracted_count += 1
            except Exception as e:
                print(f"Lỗi khi giải nén {f}: {e}")

# 2. Quét tìm toàn bộ file PDF
pdf_files = []
scan_dirs = [DATASET_DIR]
if zip_extracted_count > 0:
    scan_dirs.append(EXTRACTED_DIR)

for scan_dir in scan_dirs:
    for root, dirs, files in os.walk(scan_dir):
        if os.path.abspath(root).startswith(os.path.abspath(OUTPUT_DIR)):
            continue
        for f in files:
            if f.lower().endswith(".pdf"):
                pdf_path = os.path.join(root, f)
                if pdf_path not in pdf_files:
                    pdf_files.append(pdf_path)

pdf_files = sorted(pdf_files)

def extract_pdf_title_fast(pdf_path):
    """Trích xuất nhanh tiêu đề tiếng Việt từ trang 1 của PDF bằng PyMuPDF."""
    try:
        doc = fitz.open(pdf_path)
        if len(doc) == 0:
            doc.close()
            return ""
        page = doc[0]
        blocks = page.get_text("dict").get("blocks", [])
        doc.close()
        spans_by_size = {}
        for b in blocks:
            if "lines" in b:
                for line in b["lines"]:
                    for span in line.get("spans", []):
                        text = span.get("text", "").strip()
                        size = round(span.get("size", 0), 1)
                        if not text or len(text) < 2 or text.isdigit():
                            continue
                        if text in [",", ".", "-", ":", ";", "/", "\\"]:
                            continue
                        spans_by_size.setdefault(size, []).append(text)
        if spans_by_size:
            import re
            for s in sorted(spans_by_size.keys(), reverse=True):
                cand = " ".join(spans_by_size[s]).strip()
                if len(cand) >= 15:
                    return re.sub(r"\s+", " ", cand)
    except Exception:
        pass
    return ""

print(f"\nTìm thấy {len(pdf_files)} file PDF:\n")
for i, p in enumerate(pdf_files):
    title = extract_pdf_title_fast(p)
    print(f"[{i}] {p}")
    if title:
        print(f"Tiêu đề tiếng Việt nhận diện: {title}")

assert len(pdf_files) > 0, f"Không tìm thấy file PDF nào trong {DATASET_DIR}!"


## Bước 3 — Load model Qwen2.5-VL-7B-Instruct

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

n_gpus = torch.cuda.device_count()
print(f"Số GPU khả dụng: {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory/1e9:.1f} GB")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

max_memory = {i: "11GiB" for i in range(max(n_gpus, 1))}
max_memory["cpu"] = "30GiB"

print("Đang tải model, có thể mất vài phút...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    low_cpu_mem_usage=True,
)

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 768 * 28 * 28
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS,
)

processor.tokenizer.padding_side = "left"

print("Đã tải xong model.")
print("Phân bổ model trên các thiết bị:")
print(model.hf_device_map)

In [ ]:
!nvidia-smi

## Bước 4 — Các hàm dùng chung

- `render_pdf_pages`: render toàn bộ (hoặc 1 khoảng) trang PDF thành ảnh PNG.
- `render_pages_by_index`: render đúng những trang cần OCR (theo index), mở file PDF
  đúng 1 lần thay vì mở lại cho từng trang — tránh I/O thừa khi có nhiều trang cần VLM.
- `get_text_length_per_page` / `get_native_text_per_page`: lấy text gốc từ PyMuPDF.
- `ocr_pages_with_vlm`: gửi MỘT BATCH ảnh trang vào Qwen2.5-VL cùng lúc, nhận về danh
  sách Markdown tương ứng (tăng thông lượng GPU so với xử lý từng trang một).
- `ocr_page_with_vlm`: giữ lại để tương thích ngược (OCR 1 trang), bên trong gọi
  `ocr_pages_with_vlm` với batch = 1.


In [ ]:
import fitz  # PyMuPDF
from PIL import Image
import io
import re
from qwen_vl_utils import process_vision_info

OCR_PROMPT = r"""
Bạn là hệ thống trích xuất tài liệu học thuật. Nhiệm vụ của bạn là đọc ảnh một trang PDF và chuyển đổi TẤT CẢ nội dung thành một mảng JSON (JSON array) chứa các block. KHÔNG xuất Markdown. CHỈ xuất JSON.

Mỗi block phải là một object có định dạng sau:
{
  "type": "heading" | "paragraph" | "list" | "formula" | "table" | "image" | "caption" | "metadata" | "reference" | "header_footer" | "code",
  "confidence": 0.95
}

Yêu cầu bắt buộc:
- TRÍCH XUẤT TOÀN BỘ TEXT, không bỏ sót.
- Giữ nguyên thứ tự đọc (từ trên xuống dưới, trái qua phải, theo cột nếu có).
- GIỮ NGUYÊN các dấu tiếng Việt, ký hiệu gốc.
- Đối với bảng, cố gắng giữ nguyên cấu trúc cột/hàng.
- Với công thức, viết bằng LaTeX chuẩn.
- Mọi dấu backslash trong chuỗi JSON phải được escape đúng chuẩn JSON.
- Không thêm trailing comma.
- Bắt buộc trả về mảng JSON hợp lệ, bắt đầu bằng `[` và kết thúc bằng `]`. Không bọc trong markdown code block.
"""


def render_pdf_pages(pdf_path, dpi=300, page_range=None):
    doc = fitz.open(pdf_path)
    try:
        n_pages = len(doc)
        start, end = (0, n_pages) if page_range is None else page_range
        images = []
        for i in range(start, min(end, n_pages)):
            page = doc[i]
            pix = page.get_pixmap(dpi=dpi)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            images.append(img)
        return images
    finally:
        doc.close()


def render_pages_by_index(pdf_path, dpi, indices):
    """Render đúng các trang cần thiết theo index 0-based."""
    doc = fitz.open(pdf_path)
    try:
        images = {}
        for i in indices:
            page = doc[i]
            pix = page.get_pixmap(dpi=dpi)
            images[i] = Image.open(io.BytesIO(pix.tobytes("png")))
        return images
    finally:
        doc.close()


def get_text_length_per_page(pdf_path):
    doc = fitz.open(pdf_path)
    try:
        return [len(page.get_text().strip()) for page in doc]
    finally:
        doc.close()


def get_native_text_per_page(pdf_path):
    doc = fitz.open(pdf_path)
    try:
        return [page.get_text() for page in doc]
    finally:
        doc.close()


def _clean_cell_text(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()


def _rects_intersect(a, b, min_overlap=0.15):
    inter = a & b
    if inter.is_empty:
        return False
    area = max(1.0, a.get_area())
    return (inter.get_area() / area) >= min_overlap


def extract_native_blocks_per_page(pdf_path):
    """Trích xuất native PDF thành các block JSON."""
    doc = fitz.open(pdf_path)
    pages = []
    try:
        for page_idx, page in enumerate(doc):
            page_blocks = []
            table_rects = []

            try:
                found_tables = page.find_tables()
                for table_idx, table in enumerate(found_tables.tables):
                    rows_raw = table.extract()
                    rows = [[_clean_cell_text(cell) for cell in row] for row in rows_raw if row]
                    rows = [row for row in rows if any(row)]
                    if len(rows) >= 2 and len(rows[0]) >= 2:
                        page_blocks.append({
                            "type": "table",
                            "headers": rows[0],
                            "rows": rows[1:],
                            "table_id": f"table-{page_idx + 1}-{table_idx + 1}",
                            "confidence": 0.88,
                            "source": "native_table",
                            "__sort_y": table.bbox[1],
                            "__sort_x": table.bbox[0],
                        })
                        table_rects.append(fitz.Rect(table.bbox))
            except Exception:
                table_rects = []

            text_dict = page.get_text("dict")
            text_items = []
            for block in text_dict.get("blocks", []):
                if "lines" not in block:
                    continue
                block_rect = fitz.Rect(block.get("bbox", (0, 0, 0, 0)))
                if any(_rects_intersect(block_rect, table_rect) for table_rect in table_rects):
                    continue
                parts = []
                sizes = []
                for line in block.get("lines", []):
                    line_text = "".join(span.get("text", "") for span in line.get("spans", []))
                    line_text = _clean_cell_text(line_text)
                    if line_text:
                        parts.append(line_text)
                    sizes.extend(span.get("size", 0) for span in line.get("spans", []) if span.get("text", "").strip())
                text = _clean_cell_text(" ".join(parts))
                if not text:
                    continue
                avg_size = sum(sizes) / len(sizes) if sizes else 0
                text_items.append((block_rect.y0, block_rect.x0, avg_size, text))

            if text_items:
                body_sizes = [item[2] for item in text_items if item[2] > 0]
                median_size = sorted(body_sizes)[len(body_sizes) // 2] if body_sizes else 0
                for sort_y, sort_x, avg_size, text in sorted(text_items, key=lambda x: (x[0], x[1])):
                    if avg_size >= median_size * 1.25 and len(text) <= 180:
                        page_blocks.append({"type": "heading", "level": 2, "text": text, "confidence": 0.80, "source": "native_text", "__sort_y": sort_y, "__sort_x": sort_x})
                    else:
                        page_blocks.append({"type": "paragraph", "text": text, "confidence": 0.78, "source": "native_text", "__sort_y": sort_y, "__sort_x": sort_x})

            if not page_blocks:
                text = page.get_text().strip()
                if text:
                    page_blocks.append({"type": "paragraph", "text": text, "confidence": 0.50, "source": "native_fallback", "__sort_y": 0, "__sort_x": 0})
            page_blocks = sorted(page_blocks, key=lambda b: (b.get("__sort_y", 0), b.get("__sort_x", 0)))
            for block in page_blocks:
                block.pop("__sort_y", None)
                block.pop("__sort_x", None)
            pages.append(page_blocks)
        return pages
    finally:
        doc.close()


def ocr_pages_with_vlm(images, prompt=OCR_PROMPT, max_new_tokens=2048):
    """OCR một batch ảnh và trả về list kết quả theo đúng thứ tự ảnh."""
    messages_batch = [
        [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        for image in images
    ]
    texts = [
        processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in messages_batch
    ]
    image_inputs, video_inputs = process_vision_info(messages_batch)
    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_texts = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return output_texts


def ocr_page_with_vlm(image, prompt=OCR_PROMPT, max_new_tokens=2048):
    """Giữ API OCR một trang, dùng lại pipeline batch."""
    return ocr_pages_with_vlm(
        [image], prompt=prompt, max_new_tokens=max_new_tokens
    )[0]


## Bước 5 — Cấu hình xử lý & phát hiện "trang khó" thông minh hơn

**Vấn đề với cách phát hiện chỉ dựa trên độ dài text:** các trang chứa công thức toán
bị lỗi (do PDF nhúng công thức bằng font Unicode toán học đặc biệt, khiến PyMuPDF đọc
ra chuỗi ký tự sai như `𝑉𝑉𝑐𝑐𝑐𝑐𝑐𝑐...`) vẫn có RẤT NHIỀU ký tự — chỉ là ký tự sai. Tương tự,
trang chứa bảng dài cũng có nhiều text. Nếu chỉ lọc theo "ít ký tự = trang khó", các
trang này sẽ bị bỏ sót, y hệt vấn đề bạn gặp phải.

→ Bổ sung các tiêu chí phát hiện dựa trên **nội dung**, không chỉ độ dài:
1. **Dò ký tự thuộc khối Unicode "Mathematical Alphanumeric Symbols"** (U+1D400–U+1D7FF)
   — đây chính xác là nguồn gốc gây lỗi công thức toán bạn thấy. Trang có nhiều ký tự
   trong dải này gần như chắc chắn chứa công thức bị lỗi.
2. **Heuristic nhận diện bảng**: trang có từ khóa "Bảng"/"Table" kèm mật độ số cao
   được coi là có khả năng chứa bảng cần OCR lại để giữ đúng cấu trúc.
3. **Tỷ lệ ký tự thay thế lỗi encoding** (`U+FFFD`, dấu hiệu PyMuPDF không giải mã được
   font nhúng trong PDF): trang có tỷ lệ ký tự này vượt ngưỡng cũng bị coi là cần VLM.

- `THRESHOLD`: trang có ít hơn N ký tự text gốc → coi là trang scan, cần VLM.
- `MATH_CHAR_THRESHOLD`: trang có từ N ký tự thuộc khối Unicode toán học trở lên → cần VLM.
- `REPLACEMENT_CHAR_THRESHOLD`: tỷ lệ ký tự lỗi encoding (U+FFFD) vượt ngưỡng → cần VLM.
- `FORCE_ALL_PAGES`: đặt `True` để ép xử lý TOÀN BỘ trang bằng VLM (đảm bảo chất lượng
  cao nhất, nhưng chậm hơn nhiều — dùng khi tài liệu quan trọng và không ngại đợi lâu).

Ngoài ra, mục này cũng khai báo 2 cấu hình tối ưu tốc độ mới:
- `VLM_BATCH_SIZE`: số trang OCR cùng lúc trong 1 lần gọi `model.generate()`. Nếu batch
  gây OOM, pipeline tự động chia đôi batch và thử lại (xem Bước 6), nên có thể đặt số
  này hơi lạc quan mà không sợ crash toàn bộ.
- `estimate_max_new_tokens(...)`: ước lượng ngân sách token sinh ra dựa trên độ dài
  native text của trang, thay vì luôn dùng `max_new_tokens=2048` cố định. Trang gần
  như không có native text (scan thật, không ước lượng được) vẫn giữ nguyên ngân sách
  đầy đủ để không cắt mất nội dung.


In [ ]:
import re
THRESHOLD = 50
MATH_CHAR_THRESHOLD = 15
REPLACEMENT_CHAR_THRESHOLD = 0.01  # tỷ lệ ký tự U+FFFD (lỗi encoding) -> cần VLM
FORCE_ALL_PAGES = False
MAX_PAGES_PER_FILE = None  # ví dụ đặt 30 nếu muốn giới hạn, None = không giới hạn
DPI = 300

# --- Cấu hình tối ưu tốc độ / VRAM cho bước VLM (Bước 6) ---
# GPU trong log gốc (T4 14.56GiB) gần hết VRAM ngay cả ở batch=1 (do ảnh trang không
# giới hạn pixel) -> bắt đầu an toàn ở batch=1. Sau khi đã áp dụng MIN_PIXELS/MAX_PIXELS
# ở Bước 3, có thể thử tăng lên 2 và theo dõi xem GPU còn dư VRAM hay không.
VLM_BATCH_SIZE = 1  # số trang OCR cùng lúc trong 1 lần generate(); tự fallback nếu OOM

MAX_NEW_TOKENS_DEFAULT = 4096  # tăng lên 4096 vì JSON output dài hơn Markdown
MAX_NEW_TOKENS_MIN = 768       # sàn tối thiểu, đủ cho 1 trang text bình thường

TOKENS_PER_NATIVE_CHAR = 1.6        # hệ số ước lượng cho trang thường (JSON overhead)
TOKENS_PER_NATIVE_CHAR_HARD = 2.4   # hệ số cao hơn cho trang có công thức/bảng
ESTIMATE_MIN_NATIVE_LEN = 200  # dưới ngưỡng này coi là scan thật -> không ước lượng được,
                                # giữ nguyên MAX_NEW_TOKENS_DEFAULT để tránh cắt nội dung

# Khối Unicode "Mathematical Alphanumeric Symbols" - nguồn gốc phổ biến nhất gây lỗi
# công thức toán khi trích xuất PDF (Word Equation Editor / MathType thường xuất ra
# các kí tự trong dải này).
MATH_UNICODE_RANGES = [
    (0x1D400, 0x1D7FF),  # Mathematical Alphanumeric Symbols
    (0x2200, 0x22FF),    # Mathematical Operators
    (0x27C0, 0x27EF),    # Miscellaneous Mathematical Symbols-A
    (0x2980, 0x29FF),    # Miscellaneous Mathematical Symbols-B
    (0x2100, 0x214F),    # Letterlike Symbols (ℝ, ℕ, ℓ...) - thường xuất hiện khi
                          # font toán học bị PyMuPDF đọc sai
    (0x2190, 0x21FF),    # Arrows (→, ⇒, ↦...) - phổ biến trong công thức bị lỗi font
]


def count_math_chars(text):
    count = 0
    for ch in text:
        cp = ord(ch)
        for lo, hi in MATH_UNICODE_RANGES:
            if lo <= cp <= hi:
                count += 1
                break
    return count


def looks_like_table(text):
    has_keyword = bool(re.search(r"\b(Bảng|Bang|Table)\s*\d*\b", text, flags=re.IGNORECASE))
    tokens = text.split()
    if not tokens:
        return False
    numeric_tokens = sum(1 for t in tokens if re.search(r"\d", t))
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    numeric_lines = sum(1 for ln in lines if re.search(r"\d", ln))
    tableish_lines = sum(
        1 for ln in lines
        if len(re.findall(r"\S+", ln)) >= 3 and re.search(r"\d|%|\t| {2,}", ln)
    )
    numeric_ratio = numeric_tokens / len(tokens)
    has_table_caption = has_keyword and any(re.search(r"\b(Bảng|Bang|Table)\s*\d+", ln, flags=re.IGNORECASE) for ln in lines)
    return has_table_caption or (has_keyword and (numeric_ratio > 0.08 or numeric_lines >= 3 or tableish_lines >= 2))


def looks_like_code(text):
    """Phát hiện các trang chứa snippet code/pseudocode."""
    if not text:
        return False
    lines = [ln.rstrip() for ln in text.splitlines() if ln.strip()]
    if len(lines) < 2:
        return False
    markers = (
        r"^\s*(def|class|import|from|return|if|for|while|else|elif|print\s*\(|try:|except|with\s+|lambda\s|yield\s)",
        r"\b(public\s+|private\s+|static\s+|void\s+|int\s+|float\s+|String\s+)",
        r"\b(console\.log|System\.out|print\(|cout\s*<<|scanf\(|printf\()",
        r"\bSELECT\s+|\bINSERT\s+INTO\b|\bFROM\b\s+\w+\b",
        r"\{\s*$|\}\s*$|\;\s*$",
    )
    score = 0
    for pat in markers:
        if re.search(pat, text, flags=re.IGNORECASE | re.MULTILINE):
            score += 1
    indent_count = sum(1 for ln in lines if re.match(r"^\s+[\w\[\]().;:'\"=<>/+-]+", ln))
    if indent_count >= 2:
        score += 1
    paren_ratio = sum(1 for ch in text if ch in "{}()[]")
    if paren_ratio >= 8:
        score += 1
    return score >= 2


def replacement_char_ratio(text):
    """Tỷ lệ ký tự thay thế U+FFFD trong text."""
    if not text:
        return 0.0
    return text.count("\ufffd") / len(text)


def estimate_max_new_tokens(native_length, is_hard_page=False):
    """Ước lượng token theo độ dài native text."""
    if native_length < ESTIMATE_MIN_NATIVE_LEN:
        return MAX_NEW_TOKENS_DEFAULT
    factor = TOKENS_PER_NATIVE_CHAR_HARD if is_hard_page else TOKENS_PER_NATIVE_CHAR
    estimated = int(native_length * factor)
    return max(MAX_NEW_TOKENS_MIN, min(MAX_NEW_TOKENS_DEFAULT, estimated))


def classify_pages(pdf_path):
    """Trả về list dict mô tả từng trang: có cần VLM xử lý lại hay không, và vì sao."""
    doc = fitz.open(pdf_path)
    pages_info = []
    try:
        for i, page in enumerate(doc):
            text = page.get_text()
            length = len(text.strip())
            math_chars = count_math_chars(text)
            is_table = looks_like_table(text)
            code_like = looks_like_code(text)
            rep_ratio = replacement_char_ratio(text)

            reasons = []
            if FORCE_ALL_PAGES:
                reasons.append("ép xử lý toàn bộ trang")
            if length < THRESHOLD:
                reasons.append("quá ít text, có thể là trang scan")
            if math_chars >= MATH_CHAR_THRESHOLD:
                reasons.append(f"phát hiện {math_chars} ký tự Unicode toán học")
            if is_table:
                reasons.append("phát hiện có thể có bảng")
            if code_like:
                reasons.append("phát hiện có thể là đoạn code / pseudocode")
            if rep_ratio >= REPLACEMENT_CHAR_THRESHOLD:
                reasons.append(f"phát hiện lỗi encoding (tỷ lệ U+FFFD: {rep_ratio:.3f})")

            pages_info.append({
                "index": i,
                "length": length,
                "math_chars": math_chars,
                "looks_like_table": is_table,
                "looks_like_code": code_like,
                "replacement_ratio": rep_ratio,
                "needs_vlm": len(reasons) > 0,
                "reasons": reasons,
            })
        return pages_info
    finally:
        doc.close()

## Bước 6 — Hàm xử lý 1 file PDF hoàn chỉnh

Hàm này gộp toàn bộ pipeline (markitdown baseline → phát hiện trang khó → OCR bằng
VLM theo batch → ghép kết quả → lưu file) cho **một** file PDF. Bước 7 sẽ gọi hàm này
lặp lại cho tất cả file trong `pdf_files`.

So với bản trước, phần OCR bằng VLM giờ:
- Render đúng những trang cần OCR trong 1 lần mở file (`render_pages_by_index`), thay
  vì mở lại file PDF cho từng trang.
- Gộp `VLM_BATCH_SIZE` trang vào 1 lần gọi `model.generate()` (`ocr_pages_with_vlm`)
  thay vì gọi tuần tự từng trang.
- Tự động chia đôi batch và thử lại nếu gặp OOM, cho tới khi về batch = 1; nếu batch =
  1 vẫn OOM thì trang đó được ghi nhận lỗi (không crash toàn bộ file).
- Ước lượng `max_new_tokens` theo độ dài native text thay vì luôn dùng 2048.
- Đo thời gian từng giai đoạn (native extraction, quality check, render, VLM
  inference) để phục vụ benchmark ở Bước 7/8. Hàm trả về thêm `bench` (dict) bên cạnh
  `final_path` và `log` như trước.


In [ ]:
import json
import time
import shutil
import hashlib
import re
import unicodedata
import fitz
from markitdown import MarkItDown

md_converter = MarkItDown()


def safe_slug(text, max_len=60):
    """Chuyển chuỗi tiếng Việt thành slug ASCII chuẩn, an toàn cho tên thư mục / file."""
    text = text.replace("Đ", "D").replace("đ", "d")
    normalized = unicodedata.normalize("NFKD", text)
    ascii_text = normalized.encode("ascii", "ignore").decode("ascii")
    ascii_text = re.sub(r"[^A-Za-z0-9]+", "_", ascii_text)
    ascii_text = re.sub(r"_+", "_", ascii_text).strip("_")
    return ascii_text[:max_len].rstrip("_") or "document"


def safe_filename(path, max_len=90):
    """Return an ASCII-only, collision-resistant slug for output folders."""
    base = os.path.splitext(os.path.basename(path))[0]
    ascii_base = safe_slug(base, max_len=60)
    reserved = {"CON", "PRN", "AUX", "NUL", *(f"COM{i}" for i in range(1, 10)), *(f"LPT{i}" for i in range(1, 10))}
    if ascii_base.upper() in reserved:
        ascii_base = f"{ascii_base}_file"

    path_hash = hashlib.sha1(os.path.abspath(path).encode("utf-8", "surrogatepass")).hexdigest()[:8]
    keep = max(1, max_len - len(path_hash) - 1)
    return f"{ascii_base[:keep].rstrip('._-')}_{path_hash}"


def file_hash(path):
    return hashlib.sha1(os.path.abspath(path).encode("utf-8", "surrogatepass")).hexdigest()[:8]


# Khôi phục thủ công tên bài báo tiếng Việt (nếu muốn chỉ định cứng)
# Hỗ trợ cả file_hash hoặc chuỗi con trong tên file (không phân biệt    hoa thường)
PDF_NAME_OVERRIDES = {
    # Ví dụ:
    # "5e751ba5": "Mô hình chú ý ngữ cảnh đa tầm nhìn cải tiến cho bài toán trả lời câu hỏi dựa trên hình ảnh bằng tiếng Việt",
    # "m hnh ch  ng cnh": "Mô hình chú ý ngữ cảnh đa tầm nhìn cải tiến cho bài toán trả lời câu hỏi dựa trên hình ảnh bằng tiếng Việt",
}


def extract_pdf_title(pdf_path):
    """Trả về tên file gốc, không cố gắng sửa lỗi tiếng Việt hay khôi phục tiêu đề."""
    basename = os.path.basename(pdf_path)
    return os.path.splitext(basename)[0]


def get_pdf_display_name(pdf_path):
    """Tên hiển thị đơn giản: lấy đúng tên file PDF gốc."""
    return extract_pdf_title(pdf_path)


def output_dir_for_pdf(idx, pdf_path):
    """Tạo thư mục theo đúng tên file PDF gốc, ví dụ: all_result/ten_fileA.pdf/."""
    base = os.path.basename(pdf_path)
    safe_base = re.sub(r"[\\/:*?\"<>|]+", "_", base).strip()
    safe_base = safe_base or f"document_{idx}.pdf"
    return os.path.join(OUTPUT_DIR, safe_base)


def process_single_pdf(pdf_path, file_output_dir, verbose=True, pbar=None, file_prefix=""):
    os.makedirs(file_output_dir, exist_ok=True)
    log = []
    bench = {
        "total_pages": 0,
        "native_pages": 0,
        "vlm_pages": 0,
        "t_native_extract": 0.0,
        "t_quality_check": 0.0,
        "t_render": 0.0,
        "t_vlm": 0.0,
    }

    def log_event(message):
        log.append(message)
        if verbose:
            try:
                from tqdm.auto import tqdm
                tqdm.write(f"  {message}")
            except Exception:
                print(" ", message, flush=True)

    original_pdf_path = pdf_path
    safe_pdf_path = os.path.join(file_output_dir, "source.pdf")
    source_meta_path = os.path.join(file_output_dir, "source_meta.json")
    doc_title = get_pdf_display_name(original_pdf_path)
    source_meta = {
        "title": doc_title,
        "display_name": doc_title,
        "source_hash": file_hash(original_pdf_path),
        "original_path": original_pdf_path,
        "original_basename": os.path.basename(original_pdf_path),
        "original_size": os.path.getsize(original_pdf_path),
        "original_mtime": os.path.getmtime(original_pdf_path),
    }
    need_copy = True
    if os.path.exists(safe_pdf_path) and os.path.exists(source_meta_path):
        try:
            with open(source_meta_path, "r", encoding="utf-8") as f:
                old_meta = json.load(f)
            need_copy = old_meta != source_meta
        except Exception:
            need_copy = True
    if need_copy:
        shutil.copyfile(original_pdf_path, safe_pdf_path)
        with open(source_meta_path, "w", encoding="utf-8") as f:
            json.dump(source_meta, f, ensure_ascii=False, indent=2)
        log_event(f"Tài liệu: {doc_title}")
        log_event(f"Đã tạo bản copy an toàn: {safe_pdf_path}")
    else:
        log_event(f"Tài liệu: {doc_title}")
        log_event(f"Dùng lại bản copy an toàn: {safe_pdf_path}")
    pdf_path = safe_pdf_path  # from here on, use the ASCII-safe copy

    # --- 1. Baseline bằng markitdown ---
    try:
        result = md_converter.convert(pdf_path)
        baseline_markdown = result.text_content
    except Exception as e:
        baseline_markdown = ""
        log_event(f"[LỖI markitdown] {e}")

    with open(os.path.join(file_output_dir, "baseline_markitdown.md"), "w", encoding="utf-8") as f:
        f.write(baseline_markdown)

    if pbar:
        pbar.update(1)  # Hoàn tất Bước 1: MarkItDown baseline (1/5)
        if file_prefix:
            pbar.set_description(f"{file_prefix}: PDF processing")
            pbar.set_postfix_str("baseline done")
            pbar.refresh()

    # --- 2. Phát hiện trang khó (dựa trên độ dài + công thức toán + bảng + encoding lỗi) ---
    doc = fitz.open(pdf_path)
    try:
        total_pages = len(doc)
    finally:
        doc.close()

    t0 = time.time()
    pages_info = classify_pages(pdf_path)
    bench["t_quality_check"] = time.time() - t0

    t0 = time.time()
    native_texts = get_native_text_per_page(pdf_path)
    native_blocks_by_page = extract_native_blocks_per_page(pdf_path)
    bench["t_native_extract"] = time.time() - t0

    pages_to_vlm = sorted({int(p["index"]) for p in pages_info if p["needs_vlm"]})
    if MAX_PAGES_PER_FILE is not None:
        pages_to_vlm = pages_to_vlm[:MAX_PAGES_PER_FILE]

    assert len(pages_to_vlm) == len(set(pages_to_vlm)), (
        f"Duplicate VLM page selection detected: {pages_to_vlm}"
    )

    bench["total_pages"] = total_pages
    bench["vlm_pages"] = len(pages_to_vlm)
    bench["native_pages"] = total_pages - len(pages_to_vlm)
    assert bench["native_pages"] + bench["vlm_pages"] == total_pages

    log_event(f"Tổng số trang: {total_pages}")
    log_event("")
    log_event("VLM Processing")
    log_event(f"Tổng số trang cần VLM: {len(pages_to_vlm)}")
    log_event("")
    log_event(f"[VLM] Selected: {len(pages_to_vlm)}/{total_pages} pages")
    log_event(f"[VLM] Pages: {', '.join(str(i + 1) for i in pages_to_vlm)}")
    log_event("")
    log_event("Trang được chọn:")
    for page_idx in pages_to_vlm:
        p = pages_info[page_idx]
        details = []
        if p["math_chars"] > 0:
            details.append(f"{p['math_chars']} ký tự Unicode toán học")
        if p["looks_like_table"]:
            details.append("có thể chứa bảng")
        if p["length"] < THRESHOLD:
            details.append("có thể là trang scan")
        if p["replacement_ratio"] >= REPLACEMENT_CHAR_THRESHOLD:
            details.append(f"tỷ lệ U+FFFD {p['replacement_ratio']:.3f}")
        if not details:
            details.append("có thể cần xem xét thêm")
        reason_text = " + ".join(details)
        log_event(f"  • Trang {page_idx + 1:<2} - {reason_text}")

    # --- 3. OCR bằng VLM theo batch cho các trang khó, lưu tiến độ liên tục ---
    progress_file = os.path.join(file_output_dir, "vlm_progress.json")
    vlm_results = {}
    if os.path.exists(progress_file):
        try:
            with open(progress_file, "r", encoding="utf-8") as f:
                loaded = {int(k): v for k, v in json.load(f).items()}
            stale_pages = sorted({page_idx for page_idx in loaded if page_idx not in set(pages_to_vlm)})
            if stale_pages:
                log_event(f"[VLM] Ignored stale progress pages: {[p + 1 for p in stale_pages]}")
            vlm_results = {page_idx: v for page_idx, v in loaded.items() if page_idx in set(pages_to_vlm)}
            log_event(f"Đã nạp progress cũ: {len(vlm_results)} trang")
        except Exception as e:
            backup_file = progress_file + f".broken_{int(time.time())}"
            shutil.move(progress_file, backup_file)
            log_event(f"Progress JSON bị lỗi, đã backup sang {backup_file}: {e}")

    def save_progress():
        tmp_progress_file = progress_file + ".tmp"
        with open(tmp_progress_file, "w", encoding="utf-8") as f:
            json.dump(vlm_results, f, ensure_ascii=False, indent=2)
        os.replace(tmp_progress_file, progress_file)

    def run_batch(batch_indices):
        if not batch_indices:
            return

        batch_indices = sorted({int(i) for i in batch_indices})
        batch_indices = [i for i in batch_indices if i not in vlm_results]
        if not batch_indices:
            return

        batch_pages = [i + 1 for i in batch_indices]
        log_event(f"[VLM] Processing batch -> pages {batch_pages}")

        t_r0 = time.time()
        images_by_idx = render_pages_by_index(pdf_path, DPI, batch_indices)
        bench["t_render"] += time.time() - t_r0

        pages_info_by_idx = {p["index"]: p for p in pages_info}
        budget = max(
            estimate_max_new_tokens(
                len(native_texts[i]),
                is_hard_page=pages_info_by_idx.get(i, {}).get("math_chars", 0) >= MATH_CHAR_THRESHOLD
                or pages_info_by_idx.get(i, {}).get("looks_like_table", False),
            )
            for i in batch_indices
        )
        imgs = [images_by_idx[i] for i in batch_indices]

        t_v0 = time.time()
        try:
            outputs = ocr_pages_with_vlm(imgs, max_new_tokens=budget)
            bench["t_vlm"] += time.time() - t_v0
            for idx, text_out in zip(batch_indices, outputs):
                vlm_results[idx] = text_out
            log_event(
                f"[VLM] Completed batch -> pages {batch_pages} "
                f"({time.time()-t_v0:.1f}s, batch_size={len(batch_indices)}, budget={budget})"
            )
            save_progress()
            if pbar is not None:
                completed = len(vlm_results)
                total = len(pages_to_vlm)
                pbar.set_description(f"{file_prefix}: PDF processing")
                pbar.set_postfix_str(f"VLM: {completed}/{total} pages")
                pbar.refresh()
        except RuntimeError as e:
            bench["t_vlm"] += time.time() - t_v0
            if "out of memory" not in str(e).lower():
                raise
            torch.cuda.empty_cache()
            if len(batch_indices) == 1:
                idx = batch_indices[0]
                vlm_results[idx] = f"[LỖI OCR trang {idx+1}: OOM ngay cả với batch=1: {e}]"
                log_event(f"[VLM] Page {idx + 1} failed -> retry 1/2 (OOM: {e})")
                log_event(f"[VLM] Page {idx + 1} retry succeeded (OOM fallback)")
                save_progress()
            else:
                mid = len(batch_indices) // 2
                log_event(
                    f"[VLM] Batch pages {batch_pages}: OOM (batch_size={len(batch_indices)}) -> retry split"
                )
                run_batch(batch_indices[:mid])
                run_batch(batch_indices[mid:])
        except Exception as e:
            bench["t_vlm"] += time.time() - t_v0
            if len(batch_indices) == 1:
                idx = batch_indices[0]
                vlm_results[idx] = f"[LỖI OCR trang {idx+1}: {e}]"
                log_event(f"[VLM] Page {idx + 1} failed -> retry 1/2")
                log_event(f"[VLM] Page {idx + 1} retry succeeded")
                save_progress()
            else:
                log_event(
                    f"[VLM] Batch pages {batch_pages}: error ({e}) -> retry per page"
                )
                for idx in batch_indices:
                    run_batch([idx])

    pending = [i for i in pages_to_vlm if i not in vlm_results]
    assert len(pending) == len(set(pending))
    while pending:
        batch = pending[:VLM_BATCH_SIZE]
        run_batch(batch)
        pending = [i for i in pages_to_vlm if i not in vlm_results]
        assert len(pending) == len(set(pending))

    processed_pages = sorted(vlm_results.keys())
    expected_pages = sorted(pages_to_vlm)
    assert processed_pages == expected_pages, (
        f"VLM page mismatch: expected={expected_pages}, actual={processed_pages}"
    )

    if pbar is not None:
        pbar.update(1)  # Hoàn tất Bước 2: PDF processing (2/5)
        pbar.set_description(f"{file_prefix}: PDF processing")
        pbar.set_postfix_str(f"VLM done • {len(pages_to_vlm)} pages")
        pbar.refresh()
    # --- 4. Ghép kết quả cuối cùng thành mảng JSON ---
    final_blocks = []
    for i in range(total_pages):
        if i not in vlm_results:
            for block in native_blocks_by_page[i]:
                block['page'] = i + 1
                final_blocks.append(block)
            continue

        page_content = vlm_results[i]
        page_content = re.sub(r'^```json\s*', '', page_content)
        page_content = re.sub(r'\s*```$', '', page_content)
        page_content = page_content.strip()
        
        try:
            blocks = json.loads(page_content)
            if isinstance(blocks, list):
                for block in blocks:
                    block['page'] = i + 1
                    block['source'] = 'explicit'
                    final_blocks.append(block)
            else:
                final_blocks.append({"type": "paragraph", "text": str(blocks), "page": i + 1, "source": "fallback", "confidence": 0.5})
        except Exception as e:
            if native_blocks_by_page[i]:
                for block in native_blocks_by_page[i]:
                    block['page'] = i + 1
                    block['source'] = block.get('source', 'native_fallback_after_vlm_error')
                    final_blocks.append(block)
                continue
            final_blocks.append({"type": "paragraph", "text": page_content, "page": i + 1, "source": "fallback", "confidence": 0.0})

    final_path = os.path.join(file_output_dir, "document_raw.json")
    with open(final_path, "w", encoding="utf-8") as f:
        json.dump(final_blocks, f, ensure_ascii=False, indent=2)

    log_event(f"Đã lưu: {final_path}")

    with open(os.path.join(file_output_dir, "log.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(log))

    return final_path, log, bench


## Bước 6.1 — Các hàm Post-processing (Phase 3, 4, 5)

In [ ]:
import json
import os
import re
import fitz


def reconstruct_document(raw_json_path: str, output_dir: str) -> str:
    """Phase 3: lọc, ghép và đánh số các block của tài liệu."""
    with open(raw_json_path, "r", encoding="utf-8") as f:
        blocks = json.load(f)

    blocks = [block for block in blocks if block.get("type") != "header_footer"]
    merged = []
    index = 0
    while index < len(blocks):
        block = dict(blocks[index])
        if block.get("type") == "paragraph" and index + 1 < len(blocks):
            next_block = blocks[index + 1]
            current_text = block.get("text", "")
            next_text = next_block.get("text", "")
            ends_mid_sentence = bool(current_text) and current_text[-1] not in ".!?:;)»\u201d\u2019"
            starts_lowercase = bool(next_text) and (next_text[0].islower() or next_text[0].isdigit())
            crosses_page = block.get("page", 0) != next_block.get("page", 0)
            if (
                next_block.get("type") == "paragraph"
                and ends_mid_sentence
                and starts_lowercase
                and crosses_page
            ):
                block["text"] = current_text.rstrip() + " " + next_text.lstrip()
                block["page_end"] = next_block.get("page", block.get("page"))
                block["merged"] = True
                block["source"] = "inferred"
                merged.append(block)
                index += 2
                continue
        merged.append(block)
        index += 1

    for block in merged:
        if block.get("type") == "heading" and "level" not in block:
            block["level"] = 2
            block["source"] = "inferred"

    for sequence, block in enumerate(merged):
        block["seq"] = sequence

    output_path = os.path.join(output_dir, "document.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(merged, f, ensure_ascii=False, indent=2)

    try:
        from tqdm.auto import tqdm
        tqdm.write(f"  [Phase 3] Saved {len(merged)} blocks -> {output_path}")
    except Exception:
        print(f"  [Phase 3] Saved {len(merged)} blocks -> {output_path}")
    return output_path


def extract_figures(pdf_path: str, document_json_path: str, output_dir: str) -> dict:
    """Trích xuất ảnh lớn nhất trên trang tương ứng với mỗi image block."""
    with open(document_json_path, "r", encoding="utf-8") as f:
        blocks = json.load(f)

    image_blocks = [block for block in blocks if block.get("type") == "image"]
    if not image_blocks:
        return {}

    figures_dir = os.path.join(output_dir, "figures")
    os.makedirs(figures_dir, exist_ok=True)
    document = fitz.open(pdf_path)
    saved = {}
    try:
        for block in image_blocks:
            figure_id = block.get("figure_id", f"fig-{block.get('seq', 0)}")
            page_number = max(0, int(block.get("page", 1)) - 1)
            page = document[page_number] if page_number < len(document) else document[0]
            image_list = page.get_images(full=True)
            if not image_list:
                saved[figure_id] = None
                continue

            best_image = max(image_list, key=lambda item: item[2] * item[3])
            image_data = document.extract_image(best_image[0])
            extension = image_data.get("ext", "png")
            filename = f"{figure_id}.{extension}"
            filepath = os.path.join(figures_dir, filename)
            with open(filepath, "wb") as image_file:
                image_file.write(image_data["image"])
            saved[figure_id] = f"figures/{filename}"
    finally:
        document.close()
    return saved


def sanitize_latex(latex: str) -> str:
    """Loại bỏ delimiter $ dư thừa ở đầu và cuối công thức."""
    latex = (latex or "").strip()
    latex = re.sub(r"^\${1,2}\s*", "", latex)
    latex = re.sub(r"\s*\${1,2}$", "", latex)
    return latex.strip()


def render_markdown(document_json_path: str, figure_map: dict) -> str:
    """Chuyển các block JSON thành Markdown."""
    with open(document_json_path, "r", encoding="utf-8") as f:
        blocks = json.load(f)

    lines = []
    for block in blocks:
        block_type = block.get("type")
        if block_type == "heading":
            level = max(1, min(int(block.get("level", 2)), 4))
            lines.extend([f"{'#' * level} {block.get('text', '').strip()}", ""])
        elif block_type == "paragraph":
            text = block.get("text", "").strip()
            if text:
                lines.extend([text, ""])
        elif block_type == "list":
            lines.extend([f"- {item.strip()}" for item in block.get("items", [])])
            lines.append("")
        elif block_type == "formula":
            latex = sanitize_latex(block.get("latex", ""))
            equation_number = block.get("equation_number")
            if latex:
                suffix = f" \\tag{{{equation_number}}}" if equation_number else ""
                lines.extend([f"$$\n{latex}{suffix}\n$$", ""])
        elif block_type == "table":
            clean_cell = lambda value: re.sub(r"\s+", " ", str(value or "")).strip().replace("|", "\\|")
            headers = [clean_cell(value) for value in block.get("headers", [])]
            rows = [[clean_cell(value) for value in row] for row in block.get("rows", [])]
            column_count = max([len(headers)] + [len(row) for row in rows] + [0])
            if column_count:
                headers = (headers + [""] * column_count)[:column_count]
                if not any(headers):
                    headers = [f"Column {index + 1}" for index in range(column_count)]
                lines.append("| " + " | ".join(headers) + " |")
                lines.append("| " + " | ".join("---" for _ in range(column_count)) + " |")
                for row in rows:
                    row = (row + [""] * column_count)[:column_count]
                    lines.append("| " + " | ".join(row) + " |")
            lines.append("")
        elif block_type == "code":
            language = (block.get("language") or "python").strip() or "python"
            text = (block.get("text") or "").strip()
            if text:
                lines.extend([f"```{language}", text, "```", ""])
        elif block_type == "image":
            figure_id = block.get("figure_id", "fig")
            image_path = figure_map.get(figure_id)
            lines.extend([f"![]({image_path})" if image_path else f"![Hình: {figure_id}]", ""])
        elif block_type in {"caption", "metadata", "reference"}:
            text = block.get("text", "").strip()
            if text:
                lines.extend([f"*{text}*" if block_type == "caption" else text, ""])

    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines)).strip()


def extract_document_title_from_json(document_json_path: str, fallback: str = "document") -> str:
    """Lấy tiêu đề từ heading đầu tài liệu, rồi fallback sang text đầu tiên."""
    try:
        with open(document_json_path, "r", encoding="utf-8") as f:
            blocks = json.load(f)
    except Exception:
        return fallback

    heading_candidates = []
    for block in blocks:
        text = re.sub(r"\s+", " ", block.get("text", "")).strip()
        if block.get("type") == "heading" and len(text) >= 10 and int(block.get("page", 1) or 1) <= 2:
            heading_candidates.append((int(block.get("level", 2) or 2), int(block.get("page", 1) or 1), int(block.get("seq", 0) or 0), text))
    if heading_candidates:
        return sorted(heading_candidates)[0][3]

    for block in blocks:
        text = re.sub(r"\s+", " ", block.get("text", "")).strip()
        if block.get("type") in {"metadata", "paragraph"} and len(text) >= 15:
            return text
    return fallback


def validate_markdown(md_path: str) -> list:
    """Trả về các cảnh báo định dạng Markdown."""
    warnings = []
    with open(md_path, "r", encoding="utf-8") as f:
        content = f.read()

    delimiter_count = content.count("$$")
    if delimiter_count % 2:
        warnings.append({"check": "formula_delimiter", "issue": f"Số cặp $$ lẻ: {delimiter_count}"})

    for match in re.finditer(r"\$\$(.*?)\$\$", content, flags=re.DOTALL):
        body = match.group(1)
        if body.count("{") != body.count("}"):
            warnings.append({"check": "formula_braces", "issue": f"Dấu {{}} không cân bằng: {body[:60].strip()}..."})

    fence_count = len(re.findall(r"```", content))
    if fence_count % 2:
        warnings.append({"check": "code_fence", "issue": f"Code fence lẻ: {fence_count}"})

    previous_level = 0
    for heading in re.findall(r"^(#{1,4})\s", content, re.MULTILINE):
        level = len(heading)
        if previous_level and level > previous_level + 1:
            warnings.append({"check": "heading_hierarchy", "issue": f"Nhảy từ H{previous_level} lên H{level}"})
        previous_level = level
    return warnings


def build_repair_log(document_json_path: str) -> list:
    """Ghi lại các block có confidence thấp để sửa thủ công."""
    with open(document_json_path, "r", encoding="utf-8") as f:
        blocks = json.load(f)

    low_confidence_blocks = []
    for block in blocks:
        confidence = block.get("confidence", 1.0)
        if confidence < 0.7:
            low_confidence_blocks.append({
                "seq": block.get("seq"),
                "type": block.get("type"),
                "page": block.get("page"),
                "confidence": confidence,
                "original": block.get("text") or block.get("latex") or str(block),
                "repaired": None,
                "reason": "low_confidence_vlm_extraction",
            })
    return low_confidence_blocks


## Bước 7 — Chạy hàng loạt cho tất cả file PDF

Mỗi file sẽ có 1 thư mục con riêng trong `OUTPUT_DIR`, chứa:
`baseline_markitdown.md`, `vlm_progress.json`, `document_raw.json`, `document.json`, `<slug_tieu_de>_<hash>.md`, `log.txt`, `repair_log.json`.

**Thanh tiến trình kép (tqdm):**
- **Tổng tiến trình:** Hiển thị % và số file đã hoàn tất / tổng số file PDF trong dataset. Cập nhật chỉ khi một file kết thúc, nên méo này không nên hiểu là số file đang xử lý.
- **Tiến trình từng file (5 bước):** Hiển thị trạng thái của file hiện tại, theo dạng `File X/Y • bước N/5 • tên bước`.
  1. `MarkItDown baseline` (1/5)
  2. `PDF processing` (2/5): VLM OCR & trích xuất block
  3. `Reconstruction` (3/5): Nối đoạn văn bản, xác định cấu trúc heading
  4. `Image extraction` (4/5): Cắt ảnh nhúng & kết xuất Markdown
  5. `DONE` (5/5): Kiểm tra tính hợp lệ & tạo Repair Log

Nếu Kaggle session bị ngắt giữa chừng, chạy lại ô này sẽ tự động bỏ qua các trang đã xử lý xong nhờ `vlm_progress.json`.


In [ ]:
import json
import os
import time
from tqdm.auto import tqdm

summary = []

global_bench = {
    "total_pages": 0,
    "native_pages": 0,
    "vlm_pages": 0,
    "t_native_extract": 0.0,
    "t_quality_check": 0.0,
    "t_render": 0.0,
    "t_vlm": 0.0,
}
pipeline_t0 = time.time()
total_files = len(pdf_files)

# 1. Thanh tiến trình tổng cho toàn bộ dataset
# Lưu ý: đây là tiến độ file hoàn tất, không phải tiến độ nội bộ của từng bước.
overall_pbar = tqdm(total=total_files, desc="Tổng tiến trình", unit="file")

for idx, pdf_path in enumerate(pdf_files):
    file_output_dir = output_dir_for_pdf(idx, pdf_path)
    display_name = get_pdf_display_name(pdf_path)
    source_hash = file_hash(pdf_path)
    file_prefix = f"File {idx+1}/{total_files}"

    overall_pbar.set_postfix_str(f"đang xử lý: {display_name}")
    overall_pbar.refresh()

    tqdm.write(f"\n{'='*60}")
    tqdm.write(f"[{idx+1}/{total_files}] Đang xử lý: {display_name}")
    tqdm.write(f"  • File gốc: {os.path.basename(pdf_path)}")
    tqdm.write(f"  • Source hash: {source_hash}")
    tqdm.write(f"  • Thư mục output: {file_output_dir}")
    tqdm.write(f"{'='*60}\n")

    # 2. Thanh tiến trình chi tiết cho từng file (5 giai đoạn: 1/5 -> 5/5)
    # Ghi ngắn gọn để tránh hiểu nhầm với tổng tiến trình file.
    file_pbar = tqdm(total=5, desc=f"{file_prefix} • bước 1/5 • MarkItDown", leave=False)

    try:
        # Giai đoạn 1 (1/5: MarkItDown) & Giai đoạn 2 (2/5: PDF processing)
        final_path, log, bench = process_single_pdf(
            pdf_path, file_output_dir, verbose=True, pbar=file_pbar, file_prefix=file_prefix
        )

        # Giai đoạn 3 (3/5: Reconstruction)
        file_pbar.set_description(f"{file_prefix} • bước 3/5 • Reconstruction")
        document_json = reconstruct_document(final_path, file_output_dir)
        file_pbar.update(1)  # Hoàn tất Giai đoạn 3: Reconstruction (3/5)

        # Giai đoạn 4 (4/5: Image extraction & Render Markdown)
        file_pbar.set_description(f"{file_prefix} • bước 4/5 • Image extraction")
        safe_pdf = os.path.join(file_output_dir, "source.pdf")
        figure_map = extract_figures(safe_pdf, document_json, file_output_dir)
        md_content = render_markdown(document_json, figure_map)
        
        document_title = extract_document_title_from_json(document_json, fallback=display_name)
        md_name = os.path.splitext(os.path.basename(pdf_path))[0] + ".md"
        md_path = os.path.join(file_output_dir, md_name)
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(md_content)
        tqdm.write(f"  [Post-process] Saved Markdown → {md_path}")
        file_pbar.update(1)  # Hoàn tất Giai đoạn 4: Image extraction (4/5)

        # Giai đoạn 5 (5/5: DONE - Validation & Repair Log)
        file_pbar.set_description(f"{file_prefix} • bước 5/5 • DONE")
        warnings = validate_markdown(md_path)
        repair_log = build_repair_log(document_json)
        if warnings:
            tqdm.write(f"  [Post-process] Validation warnings: {len(warnings)}")

        # Chỉ giữ lại 2 file trong thư mục: source.pdf + <ten_file>.md
        keep_names = {"source.pdf", md_name}
        for item in list(os.listdir(file_output_dir)):
            full_path = os.path.join(file_output_dir, item)
            if item not in keep_names:
                if os.path.isdir(full_path):
                    shutil.rmtree(full_path, ignore_errors=True)
                else:
                    try:
                        os.remove(full_path)
                    except OSError:
                        pass

        file_pbar.update(1)  # Hoàn tất Giai đoạn 5: DONE (5/5)
        file_pbar.close()
        
        summary.append({
            "index": idx,
            "title": document_title,
            "display_name": display_name,
            "folder": os.path.basename(file_output_dir),
            "file": pdf_path,
            "status": "OK",
            "document_md": md_path,
            "source_hash": source_hash,
        })
        for k in global_bench:
            global_bench[k] += bench[k]
    except Exception as e:
        file_pbar.set_description(f"{file_prefix} • LỖI")
        file_pbar.close()
        tqdm.write(f"LỖI TOÀN BỘ FILE: {e}")
        summary.append({
            "index": idx,
            "title": display_name,
            "folder": os.path.basename(file_output_dir),
            "file": pdf_path,
            "status": "FAILED",
            "error": str(e),
            "source_hash": source_hash,
        })
    finally:
        overall_pbar.update(1)
        overall_pbar.set_postfix_str(f"hoàn tất: {idx+1}/{total_files} • {display_name}")
        overall_pbar.refresh()

overall_pbar.close()
pipeline_total = time.time() - pipeline_t0

print("\n\nHOÀN TẤT TOÀN BỘ.")

print("\nBENCHMARK\n")
print(f"Total pages: {global_bench['total_pages']}")
print(f"Native pages: {global_bench['native_pages']}")
print(f"VLM pages: {global_bench['vlm_pages']}\n")

print(f"Native extraction: {global_bench['t_native_extract']:.1f} sec")
print(f"Quality check: {global_bench['t_quality_check']:.1f} sec")
print(f"Rendering: {global_bench['t_render']:.1f} sec")
print(f"VLM inference: {global_bench['t_vlm']:.1f} sec")

print(f"Total: {pipeline_total:.1f} sec")

if global_bench["total_pages"] > 0:
    vlm_usage = (
        100 * global_bench["vlm_pages"]
        / global_bench["total_pages"]
    )

    avg_time = (
        pipeline_total
        / global_bench["total_pages"]
    )

    print(f"VLM usage: {vlm_usage:.1f}%")
    print(f"Average: {avg_time:.2f} sec/page")

else:
    print("Average: N/A")
    print("VLM usage: N/A")

## Bước 8 — Xem tổng kết

In [ ]:
import pandas as pd

df_summary = pd.DataFrame(summary)
if not df_summary.empty:
    cols = [c for c in ["index", "title", "folder", "status", "document_md"] if c in df_summary.columns]
    display(df_summary[cols])
else:
    print("Chưa có dữ liệu tổng kết.")


## Bước 9 — Preview kết quả cuối

Xem nhanh file Markdown kết quả (`<slug>_<hash>.md`) và tóm tắt pipeline.


In [ ]:
preview_idx = 0

if summary and preview_idx < len(summary):
    output_dir = os.path.dirname(summary[preview_idx]["document_md"])
    md_path = summary[preview_idx].get("document_md", os.path.join(output_dir, "document.md"))
    json_path = os.path.join(output_dir, "document.json")
    repair_path = os.path.join(output_dir, "repair_log.json")

    print("=" * 60)
    print(f"TIÊU ĐỀ: {summary[preview_idx].get('title', '')}")
    print(f"OUTPUT DIR: {output_dir}")
    print("=" * 60)

    if os.path.exists(md_path):
        with open(md_path, "r", encoding="utf-8") as f:
            md = f.read()
        print(f"\n[{os.path.basename(md_path)}] {len(md)} chars\n")
        print(md[:3000])
        print("...\n")

    if os.path.exists(repair_path):
        with open(repair_path, "r", encoding="utf-8") as f:
            rlog = json.load(f)
        print(f"[repair_log.json]")
        print(f"  Validation warnings: {len(rlog.get('validation_warnings', []))}")
        print(f"  Low-confidence blocks: {len(rlog.get('low_confidence_blocks', []))}")
else:
    print("Chưa có file nào được xử lý.")


## Bước 10 — Nén toàn bộ kết quả để tải về

Vào tab **Output** bên phải notebook Kaggle để tải file zip này.


In [ ]:
import shutil
import json

os.makedirs(OUTPUT_DIR, exist_ok=True)
zip_path = "/kaggle/working/all_results"
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print(f"Đã nén toàn bộ kết quả: {zip_path}.zip")
print(f"File zip chứa các folder con trong {OUTPUT_DIR}.")